# CSV source-data profiler

Run this **with your `gen_data` CSVs present** (same folder you point `data_prep` at).
It writes a single file, **`csv_profile.json`**, describing every CSV: column names + dtypes, null/unique counts, numeric ranges/sum/mean, distinct text values, and **full row dumps for the small tables** (targets, climate_scenarios, governance, banks).

Then send `csv_profile.json` back. It's synthetic data, so there's nothing sensitive in it.

Only needs `pandas` + `numpy`. Edit `DATA_PATH` in the next cell if your CSVs live elsewhere.

In [1]:
# === CSV SOURCE-DATA PROFILER =================================================
# Run this with your gen_data CSVs present. It writes ONE file: csv_profile.json
# Send that file back. It contains: per-table schema (column names + dtypes),
# null/unique counts, numeric ranges/sum/mean, categorical distinct values, and
# FULL row dumps for small tables (targets, scenarios, governance, banks, ...).
import os, json, math
from pathlib import Path
import pandas as pd
import numpy as np

# ---- CONFIG (edit DATA_PATH if your CSVs live elsewhere) --------------------
DATA_PATH          = "notebook/gen_data/"     # folder containing your *.csv files
OUTPUT_JSON        = "csv_profile.json"
FULL_DUMP_MAX_ROWS = 500             # tables with <= this many rows: every row dumped
SAMPLE_ROWS        = 5               # head/tail sample for larger tables
MAX_UNIQUE_LISTED  = 40              # cap distinct-value listing for text columns
MAX_STR_LEN        = 4000            # truncate very long cell strings

def _clean(v):
    if v is None: return None
    if isinstance(v, (np.bool_, bool)): return bool(v)
    if isinstance(v, np.integer): return int(v)
    if isinstance(v, (np.floating, float)):
        f = float(v)
        if math.isnan(f): return None
        if math.isinf(f): return "inf" if f > 0 else "-inf"
        return f
    if isinstance(v, pd.Timestamp):
        try: return v.isoformat()
        except Exception: return str(v)
    if isinstance(v, bytes):
        return v.decode("utf-8", "replace")
    if isinstance(v, str):
        return v if len(v) <= MAX_STR_LEN else v[:MAX_STR_LEN] + f"...[+{len(v)-MAX_STR_LEN} chars]"
    try:
        if pd.isna(v): return None
    except Exception:
        pass
    return v

def _row(row): return {str(k): _clean(v) for k, v in row.items()}

def profile_column(s):
    col = {"dtype": str(s.dtype), "non_null": int(s.notna().sum()), "nulls": int(s.isna().sum()),
           "n_unique": int(s.nunique(dropna=True))}
    if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
        nn = s.dropna()
        if len(nn):
            col.update(min=_clean(nn.min()), max=_clean(nn.max()),
                       mean=_clean(round(float(nn.mean()), 6)),
                       median=_clean(round(float(nn.median()), 6)),
                       sum=_clean(round(float(nn.sum()), 6)),
                       examples=[_clean(x) for x in nn.head(5).tolist()])
    else:
        vc = s.dropna().astype(str).value_counts()
        col["top_values"] = {str(k): int(v) for k, v in vc.head(MAX_UNIQUE_LISTED).items()}
        if len(vc) > MAX_UNIQUE_LISTED:
            col["top_values_note"] = f"showing {MAX_UNIQUE_LISTED} of {len(vc)} distinct values"
        col["examples"] = [_clean(x) for x in s.dropna().astype(str).head(5).tolist()]
    return col

def profile_table(name, df):
    info = {"file": name + ".csv", "rows": int(len(df)), "columns_count": int(df.shape[1]),
            "columns": list(map(str, df.columns)),
            "column_profiles": {str(c): profile_column(df[c]) for c in df.columns}}
    for key in ("bank_id", "reporting_year", "year"):
        if key in df.columns:
            try:
                info.setdefault("key_values", {})[key] = sorted(
                    [_clean(x) for x in df[key].dropna().unique().tolist()], key=lambda z: str(z))
            except Exception:
                pass
    if len(df) <= FULL_DUMP_MAX_ROWS:
        info["full_dump"] = True
        info["rows_data"] = [_row(r) for _, r in df.iterrows()]
    else:
        info["full_dump"] = False
        info["sample_head"] = [_row(r) for _, r in df.head(SAMPLE_ROWS).iterrows()]
        info["sample_tail"] = [_row(r) for _, r in df.tail(SAMPLE_ROWS).iterrows()]
    return info

def read_csv_robust(f):
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try: return pd.read_csv(f, encoding=enc)
        except UnicodeDecodeError: continue
    return pd.read_csv(f)  # let it raise with default

path = Path(DATA_PATH)
search = path if path.exists() else Path(".")
csvs = sorted(search.glob("*.csv"))
profile = {"data_path": str(search.resolve()), "n_files": len(csvs),
           "config": {"FULL_DUMP_MAX_ROWS": FULL_DUMP_MAX_ROWS, "SAMPLE_ROWS": SAMPLE_ROWS},
           "files": {}, "errors": {}}

for f in csvs:
    try:
        profile["files"][f.stem] = profile_table(f.stem, read_csv_robust(f))
    except Exception as e:
        profile["errors"][f.stem] = repr(e)

bank_ids = set()
for t in profile["files"].values():
    for b in (t.get("key_values", {}).get("bank_id") or []):
        bank_ids.add(str(b))
profile["bank_ids_across_tables"] = sorted(bank_ids)

Path(OUTPUT_JSON).write_text(json.dumps(profile, indent=2, default=str), encoding="utf-8")
size_kb = Path(OUTPUT_JSON).stat().st_size / 1024
print(f"Profiled {len(profile['files'])} CSV file(s) from {profile['data_path']}")
for name, t in profile["files"].items():
    print(f"  {name:32s} {t['rows']:>7d} rows x {t['columns_count']:>2d} cols"
          f"{'  [full dump]' if t['full_dump'] else '  [sampled]'}")
if profile["errors"]:
    print("ERRORS:", profile["errors"])
print(f"\nWrote {OUTPUT_JSON} ({size_kb:.1f} KB). Send this file back.")


Profiled 1 CSV file(s) from C:\Users\HP\Documents\IFRS_Reporting\notebooks
  financial_summary_clean               15 rows x 18 cols  [full dump]

Wrote csv_profile.json (20.9 KB). Send this file back.
